In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

print("Torch:", torch.__version__)
print("Transformers:", __import__("transformers").__version__)
print("TRL:", __import__("trl").__version__)
print("PEFT:", __import__("peft").__version__)
print("bitsandbytes:", __import__("bitsandbytes").__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/conda/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Torch: 2.9.1+cu128
Transformers: 4.57.3
TRL: 0.25.1
PEFT: 0.18.0
bitsandbytes: 0.48.2
GPU: NVIDIA GeForce RTX 4090


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME} in 4-bit on CUDA...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


Loading Qwen/Qwen2.5-Coder-7B-Instruct in 4-bit on CUDA...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [ ]:
print("Streaming Cosmopedia (stanford subset)...")
from datasets import load_dataset, IterableDataset

raw_stream = load_dataset(
    "HuggingFaceTB/cosmopedia",
    "stanford",
    split="train",
    streaming=True,
)

def is_cs_content(example):
    text = example["text"].lower()
    keywords = [
        "computer science", "algorithm", "python", "machine learning",
        "deep learning", "neural network", "data structure",
        "operating system", "database", "distributed system",
        "graph theory", "complexity", "dynamic programming",
    ]
    ban_list = ["clinical", "physiology", "anatomy", "patient", "disease"]
    if any(b in text for b in ban_list):
        return False
    return any(k in text for k in keywords)

# Filter + enforce pure {"text": str} structure
def cs_generator():
    count = 0
    for ex in raw_stream:
        if not is_cs_content(ex):
            continue
        yield {"text": ex["text"]}
        count += 1
        if count >= 5000:   # cap at 5k examples
            break

train_stream = IterableDataset.from_generator(cs_generator)

print("Dataset stream ready (up to 5k CS examples).")


Streaming Cosmopedia (stanford subset)...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Dataset stream ready (up to 5k CS examples).


In [ ]:
# Convert the streaming iterable into a list of examples (in memory)
# For 5k items this is fine on your 128 GB RAM
samples = list(train_stream)

print(f"Total filtered CS examples: {len(samples)}\n")

# Print first 10 samples (truncated)
for i, ex in enumerate(samples[:10], start=1):
    text = ex["text"].strip().replace("\n", " ")
    print(f"--- Sample {i} ---")
    print(text[:400] + ("..." if len(text) > 400 else ""))
    print()


Total filtered CS examples: 5000

--- Sample 1 ---
7.2 Dutch Pidgins and Creoles  Introduction  In this section, we will delve into the world of Dutch pidgins and creoles, examining their origins, development, structures, and functions within various communities. We will explore how these languages emerged as contact varieties between Dutch speakers and non-Dutch speaking populations, often under conditions of colonialism, slavery, or trade. Our f...

--- Sample 2 ---
2.3 The Inner Ear  Introduction ---------------------  Welcome to the fascinating world of the inner ear! This complex structure is responsible for our sense of hearing and balance, making it an essential component of our daily lives. Despite its small size (approximately 5mm in diameter), the inner ear contains some of the most intricate structures in the human body. In this section, we will delv...

--- Sample 3 ---
1.2 Importance of Electro-Neural Interfaces  As we delve deeper into the fascinating world of electro-neu

In [ ]:
from trl import SFTTrainer, SFTConfig

output_dir = "./qwen2_5_coder_cs_agent_lora"

sft_config = SFTConfig(
    output_dir=output_dir,
    dataset_text_field="text",
    max_length=1024,              # was 4096
    per_device_train_batch_size=8,    # was 4
    gradient_accumulation_steps=1,    # was 2
    learning_rate=2e-4,
    max_steps=200,
    logging_steps=10,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    save_strategy="no",
    packing=False,
)


trainer = SFTTrainer(
    model=model,
    train_dataset=train_stream,
    processing_class=tokenizer,
    args=sft_config,
    # formatting_func removed
)

print("Starting training on RTX 4090...")
trainer.train()

print("Saving LoRA adapter...")
trainer.save_model(output_dir)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training on RTX 4090...


Step,Training Loss
10,1.272000
20,1.306600
30,1.234100
40,1.189500
50,1.217500
60,1.209700
70,1.195900
80,1.220000
90,1.177200
100,1.173300


Saving LoRA adapter...


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
ADAPTER_DIR = "./qwen2_5_coder_cs_agent_lora"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading BASE model (no LoRA)...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
base_model.eval()

print("Loading FINETUNED model (base + LoRA)...")
ft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
ft_model = PeftModel.from_pretrained(ft_model, ADAPTER_DIR)
ft_model.eval()


Device: cuda
Loading BASE model (no LoRA)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading FINETUNED model (base + LoRA)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:
def generate_answer(model, prompt, max_new_tokens=2048, temperature=0.2):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
prompt = (
    "You are a CS tutor.\n\n"
    "Explain Dijkstra's algorithm step by step and show a Python implementation "
    "for finding the shortest path in a weighted graph."
)

print("=== BASE MODEL ===")
base_out = generate_answer(base_model, prompt)
print(base_out)
print("\n" + "="*80 + "\n")

print("=== FINETUNED (BASE + LoRA) MODEL ===")
ft_out = generate_answer(ft_model, prompt)
print(ft_out)


=== BASE MODEL ===
You are a CS tutor.

Explain Dijkstra's algorithm step by step and show a Python implementation for finding the shortest path in a weighted graph. Dijkstra's algorithm is used to find the shortest paths between nodes in a graph, which can be represented as a set of vertices connected by edges with associated weights. The algorithm works by maintaining a priority queue of vertices, where each vertex has an estimated distance from the source node. Initially, all distances are set to infinity except for the source node, which is set to zero. The algorithm then iteratively selects the vertex with the smallest estimated distance, updates its neighbors' distances if a shorter path is found, and adds them to the priority queue. This process continues until all vertices have been processed or the destination node is reached.
Sure! Let's go through Dijkstra's algorithm step by step and then provide a Python implementation.

### Step-by-Step Explanation of Dijkstra's Algorithm

In [ ]:
import torch, gc

def flush_cuda():
    gc.collect()                 # Python garbage collector
    if torch.cuda.is_available():
        torch.cuda.empty_cache() # Release cached GPU memory
        torch.cuda.ipc_collect() # Cleanup inter-process memory
        torch.cuda.synchronize() # Ensure all pending ops are done

flush_cuda()
print("GPU memory flushed.")


GPU memory flushed.


In [ ]:
from datasets import IterableDataset

SYSTEM_PROMPT = "You are a helpful CS tutor who explains concepts step by step."

def cs_generator_chat():
    # reuse your filtered raw_stream + is_cs_content from before
    count = 0
    for ex in raw_stream:
        if not is_cs_content(ex):
            continue
        content = ex["text"]
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Teach me this topic in detail."},
            {"role": "assistant", "content": content},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        yield {"text": text}
        count += 1
        if count >= 5000:
            break

tutor_stream = IterableDataset.from_generator(cs_generator_chat)
print(next(iter(tutor_stream))["text"][:400])


<|im_start|>system
You are a helpful CS tutor who explains concepts step by step.<|im_end|>
<|im_start|>user
Teach me this topic in detail.<|im_end|>
<|im_start|>assistant
 7.2 Dutch Pidgins and Creoles

Introduction

In this section, we will delve into the world of Dutch pidgins and creoles, examining their origins, development, structures, and functions within various communities. We will explor


In [ ]:
from trl import SFTTrainer, SFTConfig

output_dir_tutor = "./qwen2_5_coder_cs_tutor_lora"

tutor_cfg = SFTConfig(
    output_dir=output_dir_tutor,
    dataset_text_field="text",
    max_length=1024,          # was 2048/4096
    per_device_train_batch_size=2,# was 4
    gradient_accumulation_steps=1,# was 2
    learning_rate=1e-4,
    max_steps=200,                # smaller sanity run first
    logging_steps=5,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    save_strategy="no",
    packing=False,
)


tutor_trainer = SFTTrainer(
    model=model,                # your current CS-LoRA model
    train_dataset=tutor_stream,
    processing_class=tokenizer,
    args=tutor_cfg,
)

tutor_trainer.train()
tutor_trainer.save_model(output_dir_tutor)


Step,Training Loss
5,1.035200
10,1.068900
15,0.996400
20,1.010600
25,1.035200
30,0.965600
35,1.084800
40,0.971100
45,1.004900
50,1.095700


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
CS_ADAPTER_DIR = "./qwen2_5_coder_cs_agent_lora"
TUTOR_ADAPTER_DIR = "./qwen2_5_coder_cs_tutor_lora"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading shared 4-bit base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
base_model.eval()


/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/conda/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Device: cuda
Loading shared 4-bit base model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

In [ ]:
def load_adapter(model, adapter_dir):
    # Clone the base reference so we don't modify original object
    m = model
    m = PeftModel.from_pretrained(m, adapter_dir)
    m.eval()
    return m

def chat_once(model, user_msg, system_msg="You are a helpful CS tutor."):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=4096,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


In [ ]:
prompt = (
    "Explain Dijkstra's algorithm step by step and give a Python "
    "implementation for the shortest path in a weighted graph."
)

print("=== BASE MODEL ===\n")
print(chat_once(base_model, prompt))
print("\n" + "="*80 + "\n")

print("=== CS-SFT LoRA ===\n")
cs_model = load_adapter(base_model, CS_ADAPTER_DIR)
print(chat_once(cs_model, prompt))
del cs_model
torch.cuda.empty_cache()
print("\n" + "="*80 + "\n")

print("=== TUTOR-STYLE LoRA ===\n")
tutor_model = load_adapter(base_model, TUTOR_ADAPTER_DIR)
print(chat_once(tutor_model, prompt))
del tutor_model
torch.cuda.empty_cache()


=== BASE MODEL ===

system
You are a helpful CS tutor.
user
Explain Dijkstra's algorithm step by step and give a Python implementation for the shortest path in a weighted graph.
assistant
Dijkstra's Algorithm is used to find the shortest paths between nodes in a graph, which may represent, for example, road networks. The algorithm works for both directed and undirected graphs as long as all of the edge weights are non-negative.

Here is a step-by-step explanation:

1. Initialize the distance to the source node as 0 and to all other nodes as infinity.
2. Create an empty set S that will contain the visited nodes.
3. While there are unvisited nodes:
   - Pick the unvisited node with the smallest distance value, call it u.
   - Add u to the set S of visited nodes.
   - Update the distance values of the neighbors of u. If v is a neighbor of u and has not been visited yet, then check if the distance from the source to v through u is lower than the current known distance from the source to v.